## Лабораторная работа №1. Интерактивный анализ данных велопарковок SF Bay Area Bike Share в Apache Spark.
## Выполнил студент группы 6401-010302D, Стрельников Никита

## Задание

Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы. Отчёт будет отражать динамику изменения популярности языков программирования и представлять собой набор таблиц "топ-10" для каждого года.

Получившийся отчёт сохранить в формате Apache Parquet.

Для выполнения задания вы можете использовать любую комбинацию Spark API: RDD API, Dataset API, SQL API.

Импорты

In [1]:
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Row
from pyspark.sql.window import Window

In [2]:
spark = (
    SparkSession
        .builder
        .master("local[*]")
        .appName("LangsReport")
        .getOrCreate()
)
spark

#### Выборка 'posts_sample.xml'

In [3]:
posts = (
    spark
        .read
        .format('xml')
        .option('rowTag', 'row')
        .option("timestampFormat", 'y/M/d H:m:s')
        .load('posts_sample.xml')
)
posts.printSchema()

root
 |-- _AcceptedAnswerId: long (nullable = true)
 |-- _AnswerCount: long (nullable = true)
 |-- _Body: string (nullable = true)
 |-- _ClosedDate: string (nullable = true)
 |-- _CommentCount: long (nullable = true)
 |-- _CommunityOwnedDate: string (nullable = true)
 |-- _CreationDate: string (nullable = true)
 |-- _FavoriteCount: long (nullable = true)
 |-- _Id: long (nullable = true)
 |-- _LastActivityDate: string (nullable = true)
 |-- _LastEditDate: string (nullable = true)
 |-- _LastEditorDisplayName: string (nullable = true)
 |-- _LastEditorUserId: long (nullable = true)
 |-- _OwnerDisplayName: string (nullable = true)
 |-- _OwnerUserId: long (nullable = true)
 |-- _ParentId: long (nullable = true)
 |-- _PostTypeId: long (nullable = true)
 |-- _Score: long (nullable = true)
 |-- _Tags: string (nullable = true)
 |-- _Title: string (nullable = true)
 |-- _ViewCount: long (nullable = true)



In [4]:
print("Описание датасета")
posts.describe().show()

Описание датасета
+-------+--------------------+------------------+--------------------+--------------------+-----------------+--------------------+--------------------+------------------+--------------------+--------------------+--------------------+----------------------+------------------+-----------------+------------------+--------------------+------------------+------------------+-----------------+--------------------+------------------+
|summary|   _AcceptedAnswerId|      _AnswerCount|               _Body|         _ClosedDate|    _CommentCount| _CommunityOwnedDate|       _CreationDate|    _FavoriteCount|                 _Id|   _LastActivityDate|       _LastEditDate|_LastEditorDisplayName| _LastEditorUserId|_OwnerDisplayName|      _OwnerUserId|           _ParentId|       _PostTypeId|            _Score|            _Tags|              _Title|        _ViewCount|
+-------+--------------------+------------------+--------------------+--------------------+-----------------+-------------

In [5]:
print("Количество элементов: ", posts.count())

Количество элементов:  46006


#### Устанавливаем границы поиска

In [25]:
# Устанавливаем границы: 1 января 2010 года - 31 декабря 2020 года.
date_begin, date_end = "2010-01-01", "2020-12-31"
dates = (date_begin, date_end)

filtered_posts = posts.select(
    F.year(F.to_timestamp(F.col("_CreationDate"))).alias("Year"),
    F.col("_Tags")
).filter(
    (F.col("Year") >= 2010) & (F.col("Year") <= 2020) & F.col("_Tags").isNotNull()
)
filtered_posts.show(20)

+----+--------------------+
|Year|               _Tags|
+----+--------------------+
|2010|<c++><character-e...|
|2010|<sharepoint><info...|
|2010|<iphone><app-stor...|
|2010|<symfony1><schema...|
|2010|              <java>|
|2010|<visual-studio-20...|
|2010|<cakephp><file-up...|
|2010|<git><cygwin><putty>|
|2010|  <drupal><drupal-6>|
|2010|<php><wordpress><...|
|2010|<c#><winforms><da...|
|2010|<c#><asp.net><exc...|
|2010|    <sql><xml><blob>|
|2010|<.htaccess><codei...|
|2010|<wcf><web-service...|
|2010|<mod-rewrite><apa...|
|2010|<sql><database><d...|
|2010|         <ruby><rvm>|
|2010|  <android><eclipse>|
|2010|<iphone><uiimagev...|
+----+--------------------+
only showing top 20 rows


#### Выборка 'programming-languages.csv'

In [7]:
languages = (
    spark
        .read
        .format('csv')
        .option('header', 'true')
        .option("inferSchema", True)
        .load('programming-languages.csv')
        .dropna()
)

In [8]:
languages.printSchema()

root
 |-- name: string (nullable = true)
 |-- wikipedia_url: string (nullable = true)



In [9]:
print("Описание датасета")
languages.describe().show()

Описание датасета
+-------+--------+--------------------+
|summary|    name|       wikipedia_url|
+-------+--------+--------------------+
|  count|     699|                 699|
|   mean|    NULL|                NULL|
| stddev|    NULL|                NULL|
|    min|@Formula|https://en.wikipe...|
|    max|xHarbour|https://en.wikipe...|
+-------+--------+--------------------+



In [10]:
print("Количество элементов: ", languages.count())

Количество элементов:  699


#### Выводим названия языков программирования

In [11]:
language_names = [str(x[0]) for x in languages.collect()]
language_names

['A# .NET',
 'A# (Axiom)',
 'A-0 System',
 'A+',
 'A++',
 'ABAP',
 'ABC',
 'ABC ALGOL',
 'ABSET',
 'ABSYS',
 'ACC',
 'Accent',
 'Ace DASL',
 'ACL2',
 'ACT-III',
 'Action!',
 'ActionScript',
 'Ada',
 'Adenine',
 'Agda',
 'Agilent VEE',
 'Agora',
 'AIMMS',
 'Alef',
 'ALF',
 'ALGOL 58',
 'ALGOL 60',
 'ALGOL 68',
 'ALGOL W',
 'Alice',
 'Alma-0',
 'AmbientTalk',
 'Amiga E',
 'AMOS',
 'AMPL',
 'Apex (Salesforce.com)',
 'APL',
 "App Inventor for Android's visual block language",
 'AppleScript',
 'Arc',
 'ARexx',
 'Argus',
 'AspectJ',
 'Assembly language',
 'ATS',
 'Ateji PX',
 'AutoHotkey',
 'Autocoder',
 'AutoIt',
 'AutoLISP / Visual LISP',
 'Averest',
 'AWK',
 'Axum',
 'B',
 'Babbage',
 'Bash',
 'BASIC',
 'bc',
 'BCPL',
 'BeanShell',
 'Batch (Windows/Dos)',
 'Bertrand',
 'BETA',
 'Bigwig',
 'Bistro',
 'BitC',
 'BLISS',
 'Blockly',
 'BlooP',
 'Blue',
 'Boo',
 'Boomerang',
 'Bourne shell (including',
 'bash and',
 'ksh )',
 'BREW',
 'BPEL',
 'C',
 'C--',
 'C++ – ISO/IEC 14882',
 'C# – ISO/IEC

In [26]:
filtered_languages = languages.select(
    F.lower(F.trim(F.col("name"))).alias("language_key"),
    F.trim(F.col("name")).alias("Language")
).dropDuplicates(["language_key"])

#### Преобразование строки тегов вида `<java><spring><hibernate>` в отдельные теги

In [27]:
posts_tags = filtered_posts.withColumn(
    "tag",
    F.explode(
        F.split(
            F.regexp_replace(F.lower(F.col("_Tags")), r"[<>]", " "),
            r"\s+"
        )
    )
).filter(F.col("tag") != "")

#### Оставляем только те теги, которые являются языками программирования

In [28]:
language_mentions = posts_tags.join(
    F.broadcast(filtered_languages),
    posts_tags.tag == filtered_languages.language_key,
    "inner"
).select(
    "Year",
    "Language"
)

#### Подсчёт количества упоминаний языков по годам

In [33]:
year_language_counts = (
    language_mentions
        .groupBy("Year", "Language")
        .agg(F.count("*").alias("Count")
    )
)

#### Формирование топ-10 языков для каждого года

In [35]:
window_spec = Window.partitionBy("Year").orderBy(F.desc("Count"), F.asc("Language"))

top10_per_year = year_language_counts.withColumn(
    "rank",
    F.row_number().over(window_spec)
).filter(
    F.col("rank") <= 10
).select(
    "Year", "Language", "Count"
).orderBy(
    F.desc("Year"), F.desc("Count"), F.asc("Language")
)

# Выводим 50 записей
top10_per_year.show(50, truncate=False)

+----+-----------+-----+
|Year|Language   |Count|
+----+-----------+-----+
|2019|Python     |166  |
|2019|JavaScript |135  |
|2019|Java       |95   |
|2019|PHP        |65   |
|2019|R          |37   |
|2019|TypeScript |17   |
|2019|C          |14   |
|2019|Bash       |11   |
|2019|Dart       |9    |
|2019|Go         |9    |
|2018|Python     |220  |
|2018|JavaScript |198  |
|2018|Java       |146  |
|2018|PHP        |111  |
|2018|R          |66   |
|2018|TypeScript |27   |
|2018|C          |24   |
|2018|Scala      |23   |
|2018|PowerShell |13   |
|2018|Bash       |12   |
|2017|JavaScript |246  |
|2017|Java       |204  |
|2017|Python     |193  |
|2017|PHP        |138  |
|2017|R          |56   |
|2017|C          |25   |
|2017|TypeScript |20   |
|2017|Objective-C|19   |
|2017|Ruby       |17   |
|2017|Bash       |14   |
|2016|JavaScript |278  |
|2016|Java       |184  |
|2016|PHP        |155  |
|2016|Python     |146  |
|2016|R          |52   |
|2016|C          |32   |
|2016|Ruby       |24   |


#### Сохранение отчёта в Apache Parquet

In [36]:
top10_per_year.write.mode("overwrite").parquet("top10_languages_2010_2020.parquet")

In [37]:
spark.stop()